In [1]:
import polars as pl

from datetime import date

from ohlc_dss_model.data.data_loader import load_parquet
from ohlc_dss_model.data.integrity import remove_incomplete_days
from ohlc_dss_model.data.tagging import intraday_session_tagging, session_tagging

from ohlc_dss_model.features.session_aggregation import aggregate_sessions

from ohlc_dss_model.utils.dt_utils import convert_to_timezone


In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)

print(df.head(5))

shape: (5, 8)
┌──────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┬─────────────────┐
│ DateTime         ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ Session    ┆ Intraday_Sessio │
│ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        ┆ n               │
│ datetime[ns, Ame ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    ┆ date       ┆ ---             │
│ rica/New_York]   ┆         ┆         ┆         ┆         ┆        ┆            ┆ str             │
╞══════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╪═════════════════╡
│ 2016-01-03       ┆ 4592.5  ┆ 4606.75 ┆ 4592.5  ┆ 4603.25 ┆ 1134   ┆ 2016-01-04 ┆ Asia            │
│ 18:00:00 EST     ┆         ┆         ┆         ┆         ┆        ┆            ┆                 │
│ 2016-01-03       ┆ 4603.0  ┆ 4603.0  ┆ 4597.5  ┆ 4600.5  ┆ 425    ┆ 2016-01-04 ┆ Asia            │
│ 18:30:00 EST     ┆         ┆         ┆         ┆         ┆        ┆        

In [3]:
features = aggregate_sessions(df)
print(features.head(5))

shape: (5, 13)
┌────────────┬────────────┬──────────┬─────────┬───┬─────────┬────────────┬──────────┬─────────┐
│ Session    ┆ O_New York ┆ O_London ┆ O_Asia  ┆ … ┆ L_Asia  ┆ C_New York ┆ C_London ┆ C_Asia  │
│ ---        ┆ ---        ┆ ---      ┆ ---     ┆   ┆ ---     ┆ ---        ┆ ---      ┆ ---     │
│ date       ┆ f64        ┆ f64      ┆ f64     ┆   ┆ f64     ┆ f64        ┆ f64      ┆ f64     │
╞════════════╪════════════╪══════════╪═════════╪═══╪═════════╪════════════╪══════════╪═════════╡
│ 2016-01-04 ┆ 4498.75    ┆ 4529.75  ┆ 4592.5  ┆ … ┆ 4522.25 ┆ 4506.25    ┆ 4498.25  ┆ 4529.5  │
│ 2016-01-05 ┆ 4494.75    ┆ 4509.0   ┆ 4505.5  ┆ … ┆ 4498.25 ┆ 4481.5     ┆ 4494.75  ┆ 4509.25 │
│ 2016-01-06 ┆ 4396.25    ┆ 4445.0   ┆ 4482.0  ┆ … ┆ 4425.5  ┆ 4448.5     ┆ 4396.25  ┆ 4444.5  │
│ 2016-01-07 ┆ 4316.0     ┆ 4365.0   ┆ 4450.25 ┆ … ┆ 4351.5  ┆ 4293.25    ┆ 4316.75  ┆ 4365.0  │
│ 2016-01-08 ┆ 4337.0     ┆ 4339.75  ┆ 4298.0  ┆ … ┆ 4281.0  ┆ 4264.5     ┆ 4335.5   ┆ 4340.0  │
└────────────┴─

We will then add a daily candle open which is just asia open, but we will create a seperate column for it.

In [4]:
features = features.with_columns(
    pl.col("O_Asia").alias("O_ref")
)
print(features.columns)

['Session', 'O_New York', 'O_London', 'O_Asia', 'H_New York', 'H_London', 'H_Asia', 'L_New York', 'L_London', 'L_Asia', 'C_New York', 'C_London', 'C_Asia', 'O_ref']
